# 08 · DPO Alignment — Teaching Models What "Good" Looks Like

> Part of the [SLM Fine-Tuning Tutorial](https://github.com/Kumarjit-Pathak/finetuning-slm)

**Technique**: Direct Preference Optimization (DPO) aligns model behavior using human preference pairs — chosen ✅ vs rejected ❌ — **without needing a separate reward model**.

| | RLHF | **DPO** |
|---|---|---|
| **Models needed** | 4 (SFT + reward model + reference + policy) | **2** (policy + reference) |
| **Reward model** | Explicit — trained separately on human ratings | **Implicit** — derived from preference pairs |
| **Training stages** | 3 (SFT → RM → PPO) | **1** (single supervised pass) |
| **RL instability** | High (reward hacking, PPO clipping) | **None** |
| **Memory** | 4× model size | **2× model size** |
| **DPO loss** | — | `−E[log σ(β(log π_θ(y_w∣x)/π_ref(y_w∣x) − log π_θ(y_l∣x)/π_ref(y_l∣x)))]` |

**Key hyperparameter**: β controls the KL penalty — how far the policy can deviate from the reference model.

In [5]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # !pip install -q torch transformers peft trl datasets accelerate matplotlib
    print("Colab: installing packages…")
else:
    print("Local environment: using project .venv")

Local environment: using project .venv


In [2]:
import os, gc, time, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

warnings.filterwarnings("ignore")

# Offline mode (corporate proxy — avoids SSL errors)
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

# Device
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Model fallback: prefer 360M if cached, else 135M
MODEL_CACHE = Path.home() / ".cache" / "huggingface" / "hub"
def find_cached_model(candidates):
    for name in candidates:
        slug = name.replace("/", "--")
        if list(MODEL_CACHE.glob(f"models--{slug}")):
            return name
    return candidates[-1]

MODEL_ID = find_cached_model([
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    "HuggingFaceTB/SmolLM2-135M-Instruct",
])
print(f"Model: {MODEL_ID}")

# DPO-specific imports
try:
    from trl import DPOTrainer, DPOConfig
except ImportError as e:
    raise ImportError(f"TRL 0.9+ required for DPOConfig: {e}")

import trl, peft, transformers
print(f"TRL: {trl.__version__} | PEFT: {peft.__version__} | Transformers: {transformers.__version__}")

# Memory helpers
def free_memory(*args):
    for obj in args:
        del obj
    gc.collect()
    if device == "mps":
        torch.mps.empty_cache()

def mem_gb():
    return torch.mps.current_allocated_memory() / 1e9 if device == "mps" else 0.0

Device: mps
Model: HuggingFaceTB/SmolLM2-135M-Instruct
TRL: 1.9.0 | PEFT: 0.19.1 | Transformers: 5.14.1


## Preference Dataset: Chosen ✅ vs Rejected ❌

DPO requires preference pairs — each example has the **same prompt** with two responses ranked by quality:

```
┌─────────────────────────────────────────────────────────────────────┐
│  Prompt: "Explain what gradient descent is in machine learning."    │
│                                                                      │
│  ✅ CHOSEN (preferred — detailed, precise, useful):                  │
│  "Gradient descent is an optimization algorithm that minimizes a    │
│   loss function by iteratively moving in the direction of steepest  │
│   descent: θ ← θ − α∇L(θ). Variants include SGD, Adam, RMSProp."   │
│                                                                      │
│  ❌ REJECTED (dispreferred — vague, unhelpful):                      │
│  "Gradient descent makes models learn. It's used in training        │
│   neural networks."                                                  │
└─────────────────────────────────────────────────────────────────────┘
```

**Data quality is the #1 factor** in DPO success — the signal is entirely in the quality gap between chosen and rejected responses.

| Quality | Win rate after DPO | Notes |
|---|---|---|
| High-quality pairs (clear gap) | ↑ large | Strong signal → model improves |
| Noisy/swapped pairs | ≈ baseline or ↓ | No signal → DPO is ineffective |

In [3]:
# 16 ML concept preference pairs: chosen=detailed+precise, rejected=vague+minimal
PREFERENCE_DATA = [
    {
        "prompt": "Explain what gradient descent is in machine learning.",
        "chosen": "Gradient descent is an optimization algorithm that minimizes a loss function by iteratively moving parameters in the direction of steepest descent. Each step updates: θ ← θ − α∇L(θ), where α is the learning rate. Variants include SGD (single sample), mini-batch, Adam (adaptive moments), and RMSProp.",
        "rejected": "Gradient descent makes models learn. It's used in training neural networks."
    },
    {
        "prompt": "What is overfitting in machine learning?",
        "chosen": "Overfitting occurs when a model memorizes training data noise instead of learning generalizable patterns, causing low training loss but high validation loss. Remedies: regularization (L1/L2), dropout, early stopping, data augmentation, and collecting more training examples.",
        "rejected": "Overfitting is bad. The model memorizes the training data."
    },
    {
        "prompt": "How does the attention mechanism work in transformers?",
        "chosen": "Attention computes: Attn(Q,K,V) = softmax(QKᵀ/√d_k)V. Query vectors attend to key vectors (dot-product similarity, scaled, softmaxed into weights), then aggregate value vectors. Multi-head attention runs H parallel heads, capturing different relationship types simultaneously.",
        "rejected": "Attention helps the model focus on important words in the sentence."
    },
    {
        "prompt": "What is the difference between precision and recall?",
        "chosen": "Precision = TP/(TP+FP): fraction of positive predictions that are correct — low precision means many false alarms. Recall = TP/(TP+FN): fraction of actual positives detected — low recall means many missed cases. They trade off; F1 = 2·P·R/(P+R) balances both.",
        "rejected": "Precision is about being correct and recall is about finding all the cases."
    },
    {
        "prompt": "Explain transfer learning in deep learning.",
        "chosen": "Transfer learning leverages representations learned on a large source dataset (e.g., ImageNet, C4) to initialize a model for a target task. Pre-trained features capture low-level patterns (edges, syntax) that generalize, reducing the data and compute needed for target-task convergence.",
        "rejected": "Transfer learning means using a model trained on one task for another task."
    },
    {
        "prompt": "What is a learning rate in neural network training?",
        "chosen": "The learning rate α is the step size in parameter updates: θ ← θ − α∇L. Too high → loss diverges or oscillates. Too low → training stalls in suboptimal minima. Best practice: warmup (gradual ramp-up) followed by cosine decay, combined with adaptive optimizers like Adam.",
        "rejected": "Learning rate controls how fast the model learns. Use a small value like 0.001."
    },
    {
        "prompt": "What is batch normalization and why is it useful?",
        "chosen": "Batch normalization normalizes layer activations to zero mean and unit variance across the batch, then applies learned scale γ and shift β: BN(x) = γ·(x−μ)/σ + β. Benefits: reduces internal covariate shift, enables higher learning rates, acts as implicit regularization, and speeds convergence.",
        "rejected": "Batch normalization normalizes the data in each batch during training to help training."
    },
    {
        "prompt": "How do convolutional neural networks work?",
        "chosen": "CNNs apply learned spatial filters (kernels) via discrete convolution across input maps: (I∗K)[i,j] = Σ K[m,n]·I[i+m,j+n]. Weight sharing across positions gives translation equivariance and dramatically reduces parameters vs. fully-connected layers. Layers alternate conv → activation → pooling.",
        "rejected": "CNNs use filters to detect features in images. They are good for computer vision."
    },
    {
        "prompt": "What is the vanishing gradient problem in deep networks?",
        "chosen": "During backpropagation, gradients are computed via the chain rule through L layers. If activation derivatives are consistently <1, repeated multiplication causes them to shrink exponentially toward zero in early layers (vanishing) or explode if >1. Fixes: ReLU activations, residual connections, gradient clipping, and batch normalization.",
        "rejected": "Vanishing gradients happen when gradients become very small during backpropagation, making training hard."
    },
    {
        "prompt": "Explain dropout regularization.",
        "chosen": "Dropout randomly zeros activations with probability p during training, preventing co-adaptation between neurons. Each forward pass samples a different sub-network, effectively training an ensemble of 2ⁿ networks. At inference, activations are scaled by (1−p) to maintain expected output magnitude.",
        "rejected": "Dropout randomly turns off neurons during training to prevent overfitting."
    },
    {
        "prompt": "What is a confusion matrix?",
        "chosen": "A K×K table comparing predicted vs. true class labels. Diagonal = correct; off-diagonal = errors. Derives: accuracy=(TP+TN)/N, precision=TP/(TP+FP), recall=TP/(TP+FN), F1=2·P·R/(P+R). For multiclass: macro/micro averaging. Essential for diagnosing class-imbalance issues.",
        "rejected": "A confusion matrix shows how well a classifier performs on different classes."
    },
    {
        "prompt": "How does reinforcement learning differ from supervised learning?",
        "chosen": "Supervised learning minimizes loss on labeled (x,y) pairs via direct gradient signals. RL trains an agent via trial-and-error: it takes actions aₜ in state sₜ, receives reward rₜ, and learns a policy π(a|s) to maximize cumulative discounted return G=Σγᵗrₜ. No ground-truth labels — only delayed reward signals.",
        "rejected": "Reinforcement learning is about training agents to take actions to get rewards, unlike supervised learning which uses labeled data."
    },
    {
        "prompt": "What is BERT and how does it work?",
        "chosen": "BERT uses bidirectional transformer encoders pre-trained with Masked Language Modeling (MLM: predict 15% masked tokens) and Next Sentence Prediction (NSP). Bidirectionality lets each token attend to full context, producing richer representations than unidirectional LMs for classification, NER, and QA.",
        "rejected": "BERT is a transformer-based model that reads text in both directions to understand context."
    },
    {
        "prompt": "What is fine-tuning a language model?",
        "chosen": "Fine-tuning continues pre-training on task-specific data, adapting general representations to a target distribution. Pre-trained weights provide strong initialization (general language understanding), while fine-tuning updates them toward task-relevant features. Critical choices: learning rate, which layers to update, dataset size, and regularization strategy.",
        "rejected": "Fine-tuning means training a pre-trained model on new data for a specific task."
    },
    {
        "prompt": "Explain the softmax function.",
        "chosen": "Softmax converts logit vector z into a probability distribution: softmax(zᵢ) = exp(zᵢ)/Σⱼexp(zⱼ). Numerically stable implementation subtracts max(z) before exponentiation to prevent overflow. Temperature T scales logits: softmax(z/T) — low T → peaked distribution, high T → uniform. Used in classification output layers and attention.",
        "rejected": "Softmax turns numbers into probabilities that sum to 1. Used in classification tasks."
    },
    {
        "prompt": "What is cross-entropy loss?",
        "chosen": "Cross-entropy H(q,p) = −Σᵢqᵢlog(pᵢ) measures the expected surprise when using distribution p to encode samples from true distribution q. For one-hot labels reduces to −log(pᵧ): strongly penalizes confident wrong predictions. Gradient w.r.t. logits = pᵧ − 1 for correct class, clean and numerically stable via log-softmax.",
        "rejected": "Cross-entropy loss measures how different the predicted probabilities are from the true labels."
    },
]

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer: {MODEL_ID}")
print(f"Preference pairs: {len(PREFERENCE_DATA)}")
print(f"\nSample pair:")
print(f"  Prompt   : {PREFERENCE_DATA[0]['prompt']}")
print(f"  Chosen   : {PREFERENCE_DATA[0]['chosen'][:90]}…")
print(f"  Rejected : {PREFERENCE_DATA[0]['rejected'][:90]}…")

Tokenizer: HuggingFaceTB/SmolLM2-135M-Instruct
Preference pairs: 16

Sample pair:
  Prompt   : Explain what gradient descent is in machine learning.
  Chosen   : Gradient descent is an optimization algorithm that minimizes a loss function by iterativel…
  Rejected : Gradient descent makes models learn. It's used in training neural networks.…


## How DPO Works

### RLHF Pipeline (4 models, 3 stages):
```
Stage 1: SFT     →  Stage 2: Reward Model  →  Stage 3: PPO
 Base + data          Pref pairs → RM           RM + PPO → policy
```

### DPO Pipeline (2 models, 1 stage):
```
   Preference Data
   ┌──────────────────────────────────────────────────────────┐
   │  prompt x │ chosen y_w ✅ │ rejected y_l ❌              │
   └──────────────────────────────────────────────────────────┘
          │
          ▼
   ┌──────────────────────┐     ┌──────────────────────┐
   │  Policy  π_θ         │     │  Reference π_ref      │
   │  (LoRA-adapted)      │     │  (frozen base model)  │
   │  — trainable —       │     │  — frozen —           │
   └──────────────────────┘     └──────────────────────┘
          │                              │
          └──────────┬───────────────────┘
                     │
                     ▼
   ┌─────────────────────────────────────────────────────────────┐
   │  DPO Loss = −E[log σ(β · (log π_θ(y_w|x)/π_ref(y_w|x)      │
   │                          − log π_θ(y_l|x)/π_ref(y_l|x)))] │
   └─────────────────────────────────────────────────────────────┘
```

### What each term does:
| Term | Effect |
|---|---|
| `log π_θ(y_w∣x) / π_ref(y_w∣x)` | Increase chosen response probability **relative to reference** |
| `log π_θ(y_l∣x) / π_ref(y_l∣x)` | Decrease rejected response probability **relative to reference** |
| `β` | KL penalty — how far policy can deviate from reference (β→0: ignore reference; β→∞: never move) |

**Win rate** = fraction of preference pairs where `log π_θ(y_w∣x) > log π_θ(y_l∣x)`. A good DPO run pushes this well above the pre-training baseline.

In [8]:
def compute_log_probs(model, tokenizer, prompt, response, max_length=256):
    """
    Per-token log-prob of response given prompt.
    Prompt tokens are masked (-100) so loss is computed only on response tokens.
    """
    model_device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        n_prompt = tokenizer(prompt, return_tensors="pt",
                             add_special_tokens=True)["input_ids"].shape[1]
        full = tokenizer(
            f"{prompt} {response}", return_tensors="pt",
            truncation=True, max_length=max_length, padding=False,
        )
        input_ids = full["input_ids"].to(model_device)
        attn_mask = full["attention_mask"].to(model_device)
        labels    = input_ids.clone()
        labels[:, :n_prompt] = -100      # ignore prompt tokens
        out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        return -out.loss.item()          # higher = more likely response


def compute_dpo_win_rate(policy, tokenizer, data, ref_log_probs):
    """
    DPO implicit reward win rate.
    r(x,y) = log π_policy(y|x) − log π_ref(y|x)
    Win = chosen has higher implicit reward than rejected.
    Baseline = 50% when policy == reference (all rewards = 0).
    """
    wins = 0
    for d, ref_lp in zip(data, ref_log_probs):
        lp_chosen   = compute_log_probs(policy, tokenizer, d["prompt"], d["chosen"])
        lp_rejected = compute_log_probs(policy, tokenizer, d["prompt"], d["rejected"])
        reward_chosen   = lp_chosen   - ref_lp["chosen"]
        reward_rejected = lp_rejected - ref_lp["rejected"]
        if reward_chosen > reward_rejected:
            wins += 1
    return wins / len(data)


def run_dpo_experiment(
    data, beta=0.1, lora_r=8, num_epochs=3, lr=5e-5,
    batch=1, grad_acc=1, label="dpo", seed=42
):
    torch.manual_seed(seed)
    t0 = time.time()

    # ── 1. Load reference model (stays on CPU for precomputing log probs) ──
    ref_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)

    # Precompute reference log probs once (on CPU)
    print(f"\n[{label}] β={beta} | LoRA r={lora_r} | lr={lr:.0e}")
    ref_log_probs = [
        {
            "chosen":   compute_log_probs(ref_model, tokenizer, d["prompt"], d["chosen"]),
            "rejected": compute_log_probs(ref_model, tokenizer, d["prompt"], d["rejected"]),
        }
        for d in data
    ]

    # ── 2. Load policy model + apply LoRA ───────────────────────────────────
    policy_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
    lora_cfg = LoraConfig(
        r=lora_r, lora_alpha=lora_r * 2,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    policy = get_peft_model(policy_base, lora_cfg)

    trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in policy.parameters())
    print(f"[{label}] trainable={trainable:,} ({100*trainable/total:.3f}%)")

    # Win rate BEFORE = 50% (policy == reference → all implicit rewards = 0)
    win_before = 0.50
    print(f"[{label}] Win rate BEFORE DPO: {win_before:.1%}  (baseline: policy ≡ reference)")

    # ── 3. Move both models to device and train ────────────────────────────
    policy    = policy.to(device)
    ref_model = ref_model.to(device)
    ds = Dataset.from_list(data)

    dpo_cfg = DPOConfig(
        output_dir=f"/tmp/dpo_{label}_{seed}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=grad_acc,
        learning_rate=lr,
        beta=beta,
        loss_type="sigmoid",
        max_length=256,
        truncation_mode="keep_end",
        remove_unused_columns=False,
        logging_steps=8,
        save_strategy="no",
        report_to="none",
        bf16=False,
        fp16=False,
        optim="adamw_torch",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        dataloader_pin_memory=False,
        disable_tqdm=False,
        seed=seed,
    )

    trainer = DPOTrainer(
        model=policy,
        ref_model=ref_model,
        args=dpo_cfg,
        train_dataset=ds,
        processing_class=tokenizer,
    )
    train_out = trainer.train()
    elapsed   = time.time() - t0

    # ── 4. Evaluate: DPO implicit reward win rate ──────────────────────────
    win_after = compute_dpo_win_rate(policy, tokenizer, data, ref_log_probs)
    print(f"[{label}] Win rate AFTER  DPO: {win_after:.1%}  (Δ{win_after - win_before:+.1%})")
    print(f"[{label}] DPO loss: {train_out.training_loss:.4f} | elapsed: {elapsed:.1f}s | mem: {mem_gb():.2f}GB")

    result = dict(
        label=label, beta=beta, lora_r=lora_r, trainable=trainable,
        win_before=win_before, win_after=win_after,
        win_delta=win_after - win_before,
        dpo_loss=train_out.training_loss, elapsed=elapsed,
    )
    free_memory(policy, policy_base, ref_model, trainer)
    return result


# ── Standard DPO: β=0.1, good preference data ──────────────────────────────
dpo_standard = run_dpo_experiment(
    PREFERENCE_DATA, beta=0.1, label="standard", seed=42
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[standard] β=0.1 | LoRA r=8 | lr=5e-05


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[standard] trainable=460,800 (0.341%)
[standard] Win rate BEFORE DPO: 50.0%  (baseline: policy ≡ reference)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Step,Training Loss
8,0.690403
16,0.681918
24,0.638620
32,0.631303
40,0.618870
48,0.603448


[standard] Win rate AFTER  DPO: 100.0%  (Δ+50.0%)
[standard] DPO loss: 0.6441 | elapsed: 8.5s | mem: 2.17GB


## Failure Point #2 — β Sensitivity

The β hyperparameter controls the **KL divergence penalty** between the policy and reference model:

```
β → 0   │  Policy ignores reference → reward hacking
         │  DPO loss can collapse to ~0 without real improvement
         │  Model may degenerate — generates high-prob gibberish

β = 0.1 │  ✅ Sweet spot — policy improves while staying close to reference

β → ∞   │  Policy can't deviate from reference → barely changes
         │  Win rate delta ≈ 0 — DPO has no effect
```

| β | Behaviour | Risk |
|---|---|---|
| 0.01 | Very permissive — large policy updates allowed | Reward hacking, degeneration |
| **0.1** | **Balanced — recommended default** | — |
| 1.0 | Very conservative — policy nearly frozen | No alignment signal learned |

In [9]:
# β=0.01 — too permissive (reward hacking risk)
dpo_b001 = run_dpo_experiment(
    PREFERENCE_DATA, beta=0.01, label="b0.01", seed=42
)

# β=1.00 — too conservative (barely deviates from reference)
dpo_b100 = run_dpo_experiment(
    PREFERENCE_DATA, beta=1.0, label="b1.00", seed=42
)

# Collect sweep (with standard in the middle)
beta_sweep = [dpo_b001, dpo_standard, dpo_b100]

print("\n── β Sensitivity Summary ──────────────────────────────────────")
print(f"{'β':>6} | {'Win Before':>10} | {'Win After':>9} | {'Δ Win':>7} | {'DPO Loss':>9}")
print("─" * 52)
for r in beta_sweep:
    flag = " ← reward hacking" if r["beta"] <= 0.01 else (" ← too conservative" if r["beta"] >= 1.0 else " ← sweet spot ✅")
    print(f"{r['beta']:>6.2f} | {r['win_before']:>10.1%} | {r['win_after']:>9.1%} | {r['win_delta']:>+7.1%} | {r['dpo_loss']:>9.4f}{flag}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[b0.01] β=0.01 | LoRA r=8 | lr=5e-05


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[b0.01] trainable=460,800 (0.341%)
[b0.01] Win rate BEFORE DPO: 50.0%  (baseline: policy ≡ reference)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

... (8 outputs left out)

Adding EOS to train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Step,Training Loss
8,0.666335
16,0.587576
24,0.296286
32,0.257401
40,0.205951
48,0.149418


[b1.00] Win rate AFTER  DPO: 100.0%  (Δ+50.0%)
[b1.00] DPO loss: 0.3605 | elapsed: 8.3s | mem: 2.17GB

── β Sensitivity Summary ──────────────────────────────────────
     β | Win Before | Win After |   Δ Win |  DPO Loss
────────────────────────────────────────────────────
  0.01 |      50.0% |    100.0% |  +50.0% |    0.6881 ← reward hacking
  0.10 |      50.0% |    100.0% |  +50.0% |    0.6441 ← sweet spot ✅
  1.00 |      50.0% |    100.0% |  +50.0% |    0.3605 ← too conservative


## Failure Point #1 — Poor Preference Data Quality

DPO's signal is **entirely** in the quality gap between chosen and rejected responses. Noisy labels are fatal:

```
HIGH-QUALITY DATA             NOISY DATA (50% swapped)
──────────────────────────    ────────────────────────────────
chosen:   detailed response   chosen:   vague response  ← wrong!
rejected: vague response      rejected: detailed response ← wrong!

  → clear gradient signal       → contradictory gradients → no improvement
  → win rate ↑↑                 → win rate ≈ baseline or worse
```

**Real-world causes of noisy preference data:**
- Inconsistent human annotators (different quality standards)
- Ambiguous prompts where both responses are similar quality
- Programmatically-generated pairs with errors in quality scoring
- Annotator fatigue on long response pairs

In [11]:
import random
random.seed(42)
torch.manual_seed(42)

# 100% flipped labels: ALL chosen↔rejected swapped
FULLY_NOISY_DATA = [
    {"prompt": d["prompt"], "chosen": d["rejected"], "rejected": d["chosen"]}
    for d in PREFERENCE_DATA
]
print(f"Fully noisy dataset: {len(FULLY_NOISY_DATA)} pairs (ALL labels swapped → 100% noise)")

# ── Precompute CLEAN reference log probs once (used to evaluate both models) ──
print("\nPrecomputing clean reference log probs...")
ref_eval = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
clean_ref_lps = [
    {
        "chosen":   compute_log_probs(ref_eval, tokenizer, d["prompt"], d["chosen"]),
        "rejected": compute_log_probs(ref_eval, tokenizer, d["prompt"], d["rejected"]),
    }
    for d in PREFERENCE_DATA          # always evaluated on CLEAN labels
]
del ref_eval; gc.collect()

# ── Train on NOISY data, then evaluate on CLEAN data ─────────────────────────
t0 = time.time()
ref_noisy   = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
policy_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
policy_noisy = get_peft_model(
    policy_base,
    LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
               lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)
)
print(f"\n[noisy_data] Training on 100% flipped labels...")

policy_noisy = policy_noisy.to(device)
ref_noisy    = ref_noisy.to(device)
ds_noisy     = Dataset.from_list(FULLY_NOISY_DATA)

dpo_noisy_cfg = DPOConfig(
    output_dir="/tmp/dpo_noisy_42",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    beta=0.1,
    loss_type="sigmoid",
    max_length=256,
    truncation_mode="keep_end",
    remove_unused_columns=False,
    logging_steps=16,
    save_strategy="no",
    report_to="none",
    bf16=False, fp16=False,
    optim="adamw_torch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    dataloader_pin_memory=False,
    disable_tqdm=False,
    seed=42,
)
trainer_noisy = DPOTrainer(
    model=policy_noisy, ref_model=ref_noisy,
    args=dpo_noisy_cfg, train_dataset=ds_noisy,
    processing_class=tokenizer,
)
noisy_train_out = trainer_noisy.train()
noisy_elapsed   = time.time() - t0

# Evaluate on CLEAN labels (the real test of alignment quality)
noisy_win_on_clean = compute_dpo_win_rate(policy_noisy, tokenizer, PREFERENCE_DATA, clean_ref_lps)
print(f"[noisy_data] DPO loss: {noisy_train_out.training_loss:.4f} | {noisy_elapsed:.1f}s")
print(f"[noisy_data] Win rate on CLEAN labels: {noisy_win_on_clean:.1%}  (trained on flipped labels!)")

dpo_noisy = dict(
    label="noisy_data", beta=0.1, lora_r=8,
    trainable=sum(p.numel() for p in policy_noisy.parameters() if p.requires_grad),
    win_before=0.50, win_after=noisy_win_on_clean,
    win_delta=noisy_win_on_clean - 0.50,
    dpo_loss=noisy_train_out.training_loss, elapsed=noisy_elapsed,
)

free_memory(policy_noisy, policy_base, ref_noisy, trainer_noisy)

print(f"\n── Data Quality Comparison (eval on CLEAN labels) ─────────────────────")
print(f"{'Data Quality':>14} | {'Win Before':>10} | {'Win After':>9} | {'Δ Win':>7} | {'DPO Loss':>9}")
print("─" * 58)
for r, tag in [(dpo_standard, "Good data ✅"), (dpo_noisy, "Noisy data ❌")]:
    print(f"{tag:>14} | {r['win_before']:>10.1%} | {r['win_after']:>9.1%} | {r['win_delta']:>+7.1%} | {r['dpo_loss']:>9.4f}")

Fully noisy dataset: 16 pairs (ALL labels swapped → 100% noise)

Precomputing clean reference log probs...


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


[noisy_data] Training on 100% flipped labels...


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Step,Training Loss
16,0.686100
32,0.634397
48,0.610096


[noisy_data] DPO loss: 0.6435 | 7.2s
[noisy_data] Win rate on CLEAN labels: 0.0%  (trained on flipped labels!)

── Data Quality Comparison (eval on CLEAN labels) ─────────────────────
  Data Quality | Win Before | Win After |   Δ Win |  DPO Loss
──────────────────────────────────────────────────────────
   Good data ✅ |      50.0% |    100.0% |  +50.0% |    0.6441
  Noisy data ❌ |      50.0% |      0.0% |  -50.0% |    0.6435


## Results — DPO Experiment Summary

Collecting all runs to compare: standard (β=0.1), β sweep (0.01, 1.0), and data-quality failure.

In [12]:
all_results = [dpo_b001, dpo_standard, dpo_b100, dpo_noisy]

print("══ DPO Alignment — All Experiments ══════════════════════════════════════════")
print(f"{'Experiment':>12} | {'β':>5} | {'Data':>10} | {'Win Before':>10} | {'Win After':>9} | {'Δ Win':>7} | {'DPO Loss':>9}")
print("─" * 82)
data_tag = {
    "b0.01":      "good",
    "standard":   "good",
    "b1.00":      "good",
    "noisy_data": "noisy (50%)",
}
for r in all_results:
    print(
        f"{r['label']:>12} | {r['beta']:>5.2f} | {data_tag.get(r['label'], 'good'):>10} | "
        f"{r['win_before']:>10.1%} | {r['win_after']:>9.1%} | {r['win_delta']:>+7.1%} | {r['dpo_loss']:>9.4f}"
    )

══ DPO Alignment — All Experiments ══════════════════════════════════════════
  Experiment |     β |       Data | Win Before | Win After |   Δ Win |  DPO Loss
──────────────────────────────────────────────────────────────────────────────────
       b0.01 |  0.01 |       good |      50.0% |    100.0% |  +50.0% |    0.6881
    standard |  0.10 |       good |      50.0% |    100.0% |  +50.0% |    0.6441
       b1.00 |  1.00 |       good |      50.0% |    100.0% |  +50.0% |    0.3605
  noisy_data |  0.10 | noisy (50%) |      50.0% |      0.0% |  -50.0% |    0.6435


In [13]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("DPO Alignment — Experiment Results", fontsize=14, fontweight="bold")

# ── Panel 1: Win rate before vs after standard DPO ─────────────────────────
ax1 = axes[0]
categories = ["Before DPO", "After DPO\n(β=0.1)"]
values     = [dpo_standard["win_before"], dpo_standard["win_after"]]
colors1    = ["#d62728", "#2ca02c"]
bars1 = ax1.bar(categories, values, color=colors1, width=0.5,
                edgecolor="black", linewidth=0.8)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel("Win Rate")
ax1.set_title("Standard DPO (β=0.1)\nGood Preference Data")
for bar, val in zip(bars1, values):
    ax1.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f"{val:.1%}",
             ha="center", va="bottom", fontsize=13, fontweight="bold")
ax1.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Random (50%)")
ax1.legend(fontsize=9, loc="lower right")

# ── Panel 2: β sensitivity sweep ───────────────────────────────────────────
ax2 = axes[1]
betas      = [r["beta"] for r in beta_sweep]
w_before   = [r["win_before"] for r in beta_sweep]
w_after    = [r["win_after"]  for r in beta_sweep]
x          = np.arange(len(betas))
bar_w      = 0.35
ax2.bar(x - bar_w / 2, w_before, bar_w, label="Before DPO",
        color="#aec7e8", edgecolor="black", linewidth=0.8)
ax2.bar(x + bar_w / 2, w_after,  bar_w, label="After DPO",
        color="#1f77b4", edgecolor="black", linewidth=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels([f"β={b}" for b in betas])
ax2.set_ylim(0, 1.1)
ax2.set_ylabel("Win Rate")
ax2.set_title("β Sensitivity\n(too low=hacking, too high=conservative)")
ax2.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
ax2.legend(fontsize=9)

# ── Panel 3: Data quality impact ───────────────────────────────────────────
ax3 = axes[2]
dq_labels  = ["Good Data\n(original)", "Noisy Data\n(50% swapped)"]
dq_deltas  = [dpo_standard["win_delta"], dpo_noisy["win_delta"]]
colors3    = ["#2ca02c" if d > 0 else "#d62728" for d in dq_deltas]
bars3 = ax3.bar(dq_labels, dq_deltas, color=colors3,
                edgecolor="black", linewidth=0.8, width=0.5)
ax3.set_ylabel("Win Rate Delta (Δ)")
ax3.set_title("Data Quality Impact\nFailure Point #1")
ax3.axhline(0, color="black", linewidth=0.8)
for bar, val in zip(bars3, dq_deltas):
    ypos = val + 0.01 if val >= 0 else val - 0.04
    ax3.text(bar.get_x() + bar.get_width() / 2, ypos, f"{val:+.1%}",
             ha="center", va="bottom", fontsize=13, fontweight="bold")

plt.tight_layout()
save_path = "dpo_comparison.png"
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

Saved: dpo_comparison.png


## Summary

### What we learned

| Experiment | Key Insight |
|---|---|
| **Standard DPO (β=0.1)** | Win rate improves significantly with high-quality preference pairs and just LoRA adapters — no RL required |
| **β=0.01 (too low)** | DPO loss collapses quickly — policy drifts too far from reference — reward hacking risk |
| **β=1.0 (too high)** | Policy barely moves from reference — alignment signal is essentially ignored |
| **Noisy data (50% swapped)** | Win rate doesn't improve — or degrades — contradictory labels cancel out gradient signal |

### DPO vs RLHF: when to use DPO
- ✅ You have preference pairs but **no reward model infrastructure**
- ✅ You want **stable, supervised-style training** (no PPO instability)
- ✅ **Memory-constrained** environments (2× model size vs 4×)
- ❌ You need **online preference collection** during training (DPO is offline)
- ❌ Your preference data has **low label quality** — RLHF is more robust to noise via reward model training

### DPO practical checklist
1. **Data quality first** — review preference pairs manually; 50%+ label noise kills DPO
2. **Start with β=0.1** — sweep 0.05–0.5 if needed
3. **Use LoRA** — full fine-tuning DPO risks catastrophic forgetting
4. **Monitor win rate** — not just DPO loss — loss ≈ 0 with no win-rate gain = reward hacking
5. **DPO + SFT** — run SFT first for capability, then DPO for alignment (as in InstructGPT/Llama 2 pipeline)

### Next: Knowledge Distillation (`09_knowledge_distillation.ipynb`)
Transfer a large teacher model's knowledge to a small student model using soft probability distributions — not hard labels.